# CS 521 HW3 Problem 3
NetID: ws39

Each run starts from the supplied base model, using 10,000 training images and the first 1,000 test images. In Colab, select a GPU and run the cells in order.

I use the model, data helpers, L1 attack, optimizer settings, and evaluation from the [course starter](https://github.com/nalinwadhwa02/CCS521FA26HW/blob/main/hw3/q3/multi_norm_adversarial_training_cifar10.ipynb). I filled in the missing attacks and losses and added timing and checkpoint saving.


In [ ]:
%pip -q install gdown
import gdown
from pathlib import Path
DATA = Path("data")
DATA.mkdir(exist_ok=True)
for file_id, name in [
    ("17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes", "linf_base.pt"),
    ("1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe", "cifar10_train_10k.pt"),
    ("1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA", "cifar10_test_1k.pt"),
]:
    path = DATA / name
    if not path.exists():
        gdown.download(id=file_id, output=str(path), quiet=False)


In [ ]:
import json
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}
RUN_DIR = None
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
def synchronize():
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)

def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()

def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

x_train, y_train = load_data(str(DATA / "cifar10_train_10k.pt"))
x_test, y_test = load_data(str(DATA / "cifar10_test_1k.pt"))
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


In [ ]:
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut

class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)

def make_model():
    return PreActResNet18().to(device)

def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load(str(DATA / "linf_base.pt"), map_location=device))
    return model

## Attacks and evaluation
The evaluation function and L1 attack below are unchanged from the starter. All attacks run in the mode selected by the caller.

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x

def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()

def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)

def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = clip_box(x, (delta + alpha * grad.sign()).clamp(-eps, eps))
    return (x + delta).detach()

def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        radius = torch.rand(len(x), 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta).clamp_min(1e-12) * eps * radius
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1)
        delta = clip_box(x, delta)
    return (x + delta).detach()

def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)

def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()

ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}

def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

## (a) Single-norm training and (b) RAMP
RAMP uses the per-image maximum of the L-infinity and L1 cross-entropy losses, plus $0.5\,KL(p_\infty\Vert p_1)$ averaged over images classified correctly under the L-infinity attack. For each parameter tensor, the projected clean gradient is $\max(0,\cos(g_n,g_a))g_n$. The update combines this with the adversarial gradient using beta = 0.5.

In [ ]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    xa = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(xa), y)

def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    xq = attack(model, x, y, "linf")
    xr = attack(model, x, y, "l1")
    model.train()
    zq, zr = model(xq), model(xr)
    ce = torch.maximum(F.cross_entropy(zq, y, reduction="none"),
                       F.cross_entropy(zr, y, reduction="none")).mean()
    keep = zq.detach().argmax(1).eq(y)
    kl = F.kl_div(F.log_softmax(zr, 1), F.softmax(zq, 1), reduction="none").sum(1)
    pairing = (kl * keep).sum() / keep.sum().clamp_min(1)
    return ce + lam * pairing

def gradient_projection(g_n, g_a):
    dot = (g_n * g_a).sum()
    cosine = dot / (g_n.norm() * g_a.norm()).clamp_min(1e-12)
    return cosine.clamp_min(0) * g_n

def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    history = []
    for epoch in range(epochs):
        synchronize()
        epoch_start = time.perf_counter()
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        synchronize()
        row = {"epoch": epoch + 1, "loss": float(np.mean(losses)), "seconds": time.perf_counter() - epoch_start}
        history.append(row)
        print(row, flush=True)
        if RUN_DIR is not None:
            torch.save(model.state_dict(), RUN_DIR / "model.pt")
            (RUN_DIR / "training.json").write_text(json.dumps(history, indent=2))
    model.eval()
    return model

## (c) Probe-guided norm sampling

For each image, I run a one-step attack for each norm. If $s_{ip}$ is the cross-entropy from probe $p$, I sample a norm using

$$\pi_{ip}=0.9\frac{\exp(s_{ip})}{\sum_q\exp(s_{iq})}+\frac{0.1}{3}.$$

I then run a fresh 10-step attack for the sampled norm and train on that example:

$$L=\frac1n\sum_i CE(f_\theta(A_{p_i}^{10}(x_i)),y_i),\qquad p_i\sim\mathrm{Categorical}(\pi_i).$$

Images with the same sampled norm are attacked together, then put back in batch order. The probe scores and sampling probabilities are detached.

The idea is to spend more attack steps on each image's weak norm without running three full attacks. The uniform part keeps every norm in the mix. I use ordinary adversarial cross-entropy, without RAMP's KL term or gradient projection. A one-step probe may pick the wrong norm, so I compare this rule with uniform sampling.

For the ablation, `uniform=True` sets all probabilities to 1/3. I still run the probes to keep the rest of the computation the same.


In [ ]:
def my_loss(model, x, y, uniform=False):
    # Probe all norms cheaply, then spend ten attack steps on one norm per image.
    model.eval()
    with torch.enable_grad():
        scores = []
        for norm in NORMS:
            probe = attack(model, x, y, norm, steps=1)
            with torch.no_grad():
                scores.append(F.cross_entropy(model(probe), y, reduction="none"))
    scores = torch.stack(scores, 1)
    probs = 0.9 * scores.softmax(1) + 0.1 / len(NORMS)
    if uniform:
        probs = torch.ones_like(probs) / len(NORMS)
    # Sampling on CPU also works on MPS; the generator is seeded by adv_train.
    selected = torch.multinomial(probs.cpu(), 1).squeeze(1).to(x.device)
    xa = torch.empty_like(x)
    for j, norm in enumerate(NORMS):
        mask = selected == j
        if mask.any():
            xa[mask] = attack(model, x[mask], y[mask], norm)
    model.train()
    return F.cross_entropy(model(xa), y)

In [ ]:
def save_json(path, value):
    path.write_text(json.dumps(value, indent=2))

def run_one(name, seed, loss_fn, epochs=10, **kwargs):
    global RUN_DIR
    RUN_DIR = RESULTS / f"{name}_seed{seed}"
    RUN_DIR.mkdir(exist_ok=True)
    model = adv_train(load_base_model(), loss_fn, epochs=epochs, seed=seed, **kwargs)
    acc = test_model(model)
    save_json(RUN_DIR / "evaluation.json", acc)
    return acc

base = test_model(load_base_model())
save_json(RESULTS / "base.json", base)
for norm in NORMS:
    run_one(norm, 0, single_norm_loss, norm=norm)
for seed in (0, 1):
    run_one("ramp", seed, ramp_loss, gp_beta=0.5)
for seed in (0, 1):
    run_one("mine", seed, my_loss)
run_one("uniform", 0, my_loss, uniform=True)


## Training time and stronger attacks
If my method is slower, I rerun RAMP from the base model for enough whole epochs to match the mean training time. I also evaluate the better of my two seeds, chosen by default union accuracy, with 200 steps and 5 restarts. The classifier has no test-time randomness or input transformation.


In [ ]:
def read(path):
    return json.loads(path.read_text())
def seconds(name, seed):
    return sum(r['seconds'] for r in read(RESULTS/f'{name}_seed{seed}'/'training.json'))

mine_time = sum(seconds('mine',s) for s in (0,1))/2
ramp_time = sum(seconds('ramp',s) for s in (0,1))/2
comparison={'mine_mean_total_seconds': mine_time, 'ramp_mean_total_seconds': ramp_time}
if mine_time > ramp_time:
    # Restart from the base and retain the default LR schedule for the longer run.
    epochs=math.ceil(10*mine_time/ramp_time)
    comparison['ramp_matched_epochs']=epochs
    for seed in (0,1):
        out=RESULTS/f'ramp_matched_seed{seed}'
        out.mkdir(exist_ok=True)
        RUN_DIR=out
        if not (out/'evaluation.json').exists():
            model=adv_train(load_base_model(),ramp_loss,epochs=epochs,gp_beta=.5,seed=seed)
            result=test_model(model)
            (out/'evaluation.json').write_text(json.dumps(result,indent=2))
        comparison[f'ramp_matched_seed{seed}_seconds']=sum(r['seconds'] for r in read(out/'training.json'))
(RESULTS/'time_comparison.json').write_text(json.dumps(comparison,indent=2))
# Predeclared selection: the better of the two full custom-method runs.
best=max((RESULTS/f'mine_seed{s}' for s in (0,1)),key=lambda d:read(d/'evaluation.json')['union'])
model=load_base_model()
model.load_state_dict(torch.load(best/'model.pt',map_location=device))
strong_path=best/'strong_evaluation.json'
if not strong_path.exists():
    print('STRONG EVALUATION',best.name,flush=True)
    original_attack = attack
    progress = {"calls": 0}
    def progress_attack(model, x, y, norm, steps=10):
        xa = original_attack(model, x, y, norm, steps)
        progress["calls"] += 1
        if progress["calls"] % 5 == 0:
            print(f"Strong evaluation: {progress['calls']}/60 attack calls, finished {norm}", flush=True)
        return xa
    attack = progress_attack
    try:
        result=test_model(model,steps=200,restarts=5)
    finally:
        attack = original_attack
    strong_path.write_text(json.dumps(result,indent=2))
(RESULTS/'strong_selection.json').write_text(json.dumps({'model':best.name,'selection':'highest default union accuracy among the two custom-method seeds','steps':200,'restarts':5},indent=2))
print('ALL EXPERIMENTS COMPLETE',flush=True)


In [ ]:
import pandas as pd
rows = [{"model": "base", **read(RESULTS / "base.json")}]
for folder in sorted(RESULTS.iterdir()):
    if folder.is_dir() and (folder / "evaluation.json").exists():
        h = read(folder / "training.json")
        rows.append({"model": folder.name, **read(folder / "evaluation.json"),
                     "seconds/epoch": np.mean([r["seconds"] for r in h])})
display(pd.DataFrame(rows))
for method in ("ramp", "mine"):
    pair = [read(RESULTS / f"{method}_seed{s}" / "evaluation.json") for s in (0,1)]
    print(method, "two-seed mean:", {k: np.mean([r[k] for r in pair]) for k in pair[0]})


## Local results

I ran these experiments on Apple M3 (MPS) with PyTorch 2.8.0. I have not run them on Colab. The code cells have no saved outputs; these tables come from the local runs. Default evaluation uses 50 steps and 2 restarts. Training times exclude evaluation and checkpoint saving.

| Model | Clean | L-inf | L2 | L1 | Union | Seconds/epoch |
|---|---:|---:|---:|---:|---:|---:|
| Base | 78.70 | 48.30 | 59.80 | 22.90 | 22.90 | — |
| linf_seed0 | 78.50 | 44.70 | 56.40 | 20.50 | 20.50 | 55.14 |
| l2_seed0 | 82.40 | 32.70 | 61.70 | 28.90 | 24.70 | 58.82 |
| l1_seed0 | 84.10 | 0.00 | 0.00 | 26.00 | 0.00 | 57.10 |
| ramp_seed0 | 78.90 | 41.90 | 61.70 | 45.40 | 39.80 | 124.72 |
| ramp_seed1 | 78.20 | 42.60 | 61.70 | 45.80 | 40.00 | 136.30 |
| mine_seed0 | 79.70 | 41.50 | 61.80 | 46.20 | 39.50 | 77.40 |
| mine_seed1 | 78.80 | 41.60 | 62.40 | 46.20 | 39.30 | 84.92 |
| uniform_seed0 | 79.40 | 40.40 | 61.90 | 47.90 | 38.80 | 70.69 |
| ramp (two-seed mean) | 78.55 | 42.25 | 61.70 | 45.60 | 39.90 | 130.51 |
| mine (two-seed mean) | 79.25 | 41.55 | 62.10 | 46.20 | 39.40 | 81.16 |

Mean complete training time: mine 811.59 s; RAMP 1305.11 s.

### Stronger evaluation
Selected model: `mine_seed0` (higher default union accuracy of the two custom-method runs).

| Attack setting | Clean | L-inf | L2 | L1 | Union |
|---|---:|---:|---:|---:|---:|
| 50 steps, 2 restarts | 79.70 | 41.50 | 61.80 | 46.20 | 39.50 |
| 200 steps, 5 restarts | 79.70 | 41.40 | 61.70 | 44.40 | 38.70 |

### Epoch training times (seconds)

| Epoch | linf_seed0 | l2_seed0 | l1_seed0 | ramp_seed0 | ramp_seed1 | mine_seed0 | mine_seed1 | uniform_seed0 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 58.71 | 60.73 | 56.40 | 114.55 | 113.71 | 73.22 | 88.58 | 70.57 |
| 2 | 55.88 | 60.96 | 57.04 | 119.97 | 144.75 | 72.25 | 89.02 | 73.57 |
| 3 | 50.62 | 64.58 | 60.52 | 136.78 | 138.26 | 71.54 | 76.29 | 69.88 |
| 4 | 55.79 | 57.46 | 58.64 | 142.39 | 137.90 | 71.27 | 70.48 | 70.66 |
| 5 | 56.16 | 57.86 | 56.34 | 143.56 | 137.63 | 71.33 | 72.52 | 70.78 |
| 6 | 59.58 | 57.49 | 56.50 | 142.95 | 141.01 | 71.00 | 89.03 | 70.62 |
| 7 | 55.79 | 57.70 | 58.03 | 117.28 | 142.09 | 73.54 | 91.32 | 69.89 |
| 8 | 55.92 | 57.47 | 55.48 | 109.70 | 139.00 | 88.94 | 90.79 | 70.15 |
| 9 | 49.96 | 59.17 | 57.19 | 111.26 | 136.97 | 90.71 | 91.05 | 70.66 |
| 10 | 52.95 | 54.74 | 54.82 | 108.77 | 131.71 | 90.24 | 90.06 | 70.10 |